### Name : Nimbalkar Sujal Sunil
### Roll No : 24207146

In [18]:
import pandas as pd

In [19]:
df = pd.read_csv("movie_metadata.csv")

In [20]:
df.head()

,color,director_name,num_critic_for_reviews,duration,director_facebook_likes,actor_3_facebook_likes,actor_2_name,actor_1_facebook_likes,gross,genres,...,num_user_for_reviews,language,country,content_rating,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes
0,Color,James Cameron,723.0,178.0,0.0,855.0,Joel David Moore,1000.0,760505847.0,Action|Adventure|Fantasy|Sci-Fi,...,3054.0,English,USA,PG-13,237000000.0,2009.0,936.0,7.9,1.78,33000
1,Color,Gore Verbinski,302.0,169.0,563.0,1000.0,Orlando Bloom,40000.0,309404152.0,Action|Adventure|Fantasy,...,1238.0,English,USA,PG-13,300000000.0,2007.0,5000.0,7.1,2.35,0
2,Color,Sam Mendes,602.0,148.0,0.0,161.0,Rory Kinnear,11000.0,200074175.0,Action|Adventure|Thriller,...,994.0,English,UK,PG-13,245000000.0,2015.0,393.0,6.8,2.35,85000
3,Color,Christopher Nolan,813.0,164.0,22000.0,23000.0,Christian Bale,27000.0,448130642.0,Action|Thriller,...,2701.0,English,USA,PG-13,250000000.0,2012.0,23000.0,8.5,2.35,164000
4,NaN,Doug Walker,NaN,NaN,131.0,NaN,Rob Walker,131.0,NaN,Documentary,...,NaN,NaN,NaN,NaN,NaN,NaN,12.0,7.1,NaN,0


In [21]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

In [22]:
df = df.drop(columns=[
'color',
'director_facebook_likes',
'actor_1_facebook_likes',
'actor_2_facebook_likes',
'actor_3_facebook_likes',
'movie_facebook_likes',
'aspect_ratio'
])

In [23]:
df = df.dropna()

print(df.shape)

(3804, 21)


In [24]:
features = ['movie_title', 'genres','director_name','actor_1_name','actor_2_name','language']
df = df[features + ['imdb_score']]

In [25]:
df['content'] = df['movie_title'] + " " + df['genres'] + " " + df['director_name'] + " " + df['actor_1_name'] + " " + df['actor_2_name'] + " " + df['language']

# Naive bayes for Content-Based Filtering

In [26]:
df['rating_class'] = df['imdb_score'].apply(lambda x: 1 if x >= 7 else 0)

X_train, X_test, y_train, y_test = train_test_split(X, df['rating_class'], test_size=0.2, random_state=42)

nb = MultinomialNB()

nb.fit(X_train, y_train)

accuracy = nb.score(X_test, y_test)

print("Naive Bayes Accuracy:", accuracy)

Naive Bayes Accuracy: 0.7003942181340341


In [27]:
vectorizer = CountVectorizer()

X = vectorizer.fit_transform(df['content'])

# KNN - For Collaborative Filtering

In [28]:
knn = NearestNeighbors(metric='cosine', algorithm='brute')

knn.fit(X)

NearestNeighbors(algorithm='brute', metric='cosine')

In [29]:
def recommend_movies_by_rating(genre, top_n=10):

    genre_movies = df[df['genres'].str.contains(genre, case=False)]

    if genre_movies.empty:
        print("No movies found for this genre")
        return

    genre_movies = genre_movies.sort_values(by='imdb_score', ascending=False)

    print(f"\nTop {top_n} {genre} Movies:\n")

    for i, row in genre_movies.head(top_n).iterrows():
        print(row['movie_title'], "-", row['imdb_score'], "-", row['genres'])

In [30]:
genre = input("Enter movie genre: ")

recommend_movies_by_rating(genre)

Enter movie genre: Drama

Top 10 Drama Movies:

The Shawshank Redemption  - 9.3 - Crime|Drama
The Godfather  - 9.2 - Crime|Drama
The Dark Knight  - 9.0 - Action|Crime|Drama|Thriller
The Godfather: Part II  - 9.0 - Crime|Drama
Schindler's List  - 8.9 - Biography|Drama|History
The Lord of the Rings: The Return of the King  - 8.9 - Action|Adventure|Drama|Fantasy
Pulp Fiction  - 8.9 - Crime|Drama
Forrest Gump  - 8.8 - Comedy|Drama
The Lord of the Rings: The Fellowship of the Ring  - 8.8 - Action|Adventure|Drama|Fantasy
Fight Club  - 8.8 - Drama
